### Овсянникова Майя группа 241, 24.02.2026

### Условие задачи

**1.** Найти любой набор данных с привязкой к географии и нанести какой-нибудь признак на интерактивную географическую карту. Размер точки должен быть отдельным атрибутом.
(!) Важно: изучите документацию и поймите, какие данные должны быть в наборе, чтобы их можно было нанести на карту, от этого будет зависеть пригодность найденного датасета для этой задачи.

**2.** Построить 3 любых графика с библиотекой `Plotly`.

### Теория

Были загружены и обработаны для дальнейших манипуляций данные. Была построена интерактивная карта инструментами библиотеки `Plotly`.

### Документация

Используемые библиотеки:
- pandas – основная библиотека для работы с табличными данными
- plotly.express – интерфейс библиотеки Plotly для быстрого создания интерактивных графиков
- plotly.graph_objects –  интерфейс Plotly для более тонкой настройки графиков. Используется для добавления хороплетов

Методы:

**A. Создание точечной карты (`plotly.express.scatter_geo`)**
Основная визуализация строится с помощью `px.scatter_geo` – интерактивная карта:
- `lat='lat'` – указывает столбец с широтой для позиционирования точек
- `lon='lng'` – указывает столбец с долготой
- `size='population'` – размер каждой точки пропорционален населению города
- `color='country'` – точки окрашиваются по странам
- `color_discrete_map=country_color_map` – использует созданный вручную словарь для назначения конкретных цветов каждой стране
- `projection='natural earth'` – устанавливает проекцию карты
- `category_orders={"country": sorted_countries}` – задает порядок стран в легенде (от самой населенной к менее населенной, согласно самому крупному городу)

**B. Настройка внешнего вида карты (`fig.update_geos`)**
- `showcountries=True`, `countrycolor='black'` – отображает границы стран
- `showland=True`, `landcolor='lightgrey'` – закрашивает сушу
- `showocean=True`, `oceancolor='grey'` – закрашивает океаны

**C. Для закраски стран (`plotly.graph_objects.Choropleth`)**
- `go.Choropleth(...)` – создает хороплет (закрашенную область)
- `locations=[country]` – указывает, какую страну закрасить
- `locationmode='country names'` – говорит, что мы передаем названия стран
- `z=[1]` – задаёт значение цвета
- `colorscale=[[0, country_color_map[country]], [1, country_color_map[country]]]` – создает цветовую шкалу из одного цвета (цвета, соответствующего стране)
- `showscale=False` – отключает отображение цветовой шкалы
- `fig.add_trace(...)` добавляет этот хороплет на основной график

**D. Настройка макета (`fig.update_layout`)**
- `legend=dict(...)` – настраивает отображение легенды
- `title=dict(text='...')` – добавляет поясняющий заголовок к легенде
- `itemsizing='constant'` – делает значки в легенде одинакового размера
- `width=1200`, `height=700` – задает размеры итогового графика

### Решение

#### Задание 1

In [30]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import nbformat

df = pd.read_csv('worldcities.csv')


In [31]:
df = df[['city', 'lat', 'lng', 'country', 'population']].dropna()

df = df[df['population'] > 2500000].copy()

df

,city,lat,lng,country,population
0,Tokyo,35.6870,139.7495,Japan,37785000.0
1,Jakarta,-6.1750,106.8275,Indonesia,33756000.0
2,Delhi,28.6100,77.2300,India,32226000.0
3,Guangzhou,23.1300,113.2600,China,26940000.0
4,Mumbai,19.0761,72.8775,India,24973000.0
...,...,...,...,...,...
321,Dingxi,35.6080,104.5920,China,2524097.0
322,Shaoxing,30.0511,120.5833,China,2521964.0
323,Yantai,37.4646,121.4478,China,2511053.0
324,Huizhou,23.1120,114.4160,China,2509243.0


In [32]:
country_max_pop = df.groupby('country')['population'].max().sort_values(ascending=False)
sorted_countries = country_max_pop.index.tolist()

unique_countries = df['country'].unique()
colors = px.colors.qualitative.Alphabet * (len(unique_countries) // 26 + 1)
country_color_map = {country: colors[i] for i, country in enumerate(unique_countries)}

In [33]:
fig = px.scatter_geo(
    df,
    lat='lat',
    lon='lng',
    size='population',
    color='country',
    color_discrete_map=country_color_map, 
    title='Города мира с населением > 2 500 000',
    projection='natural earth',
    opacity=1,
    category_orders={"country": sorted_countries} 
)

fig.update_geos(
    showcountries=True,
    countrycolor="black", 
    countrywidth=0.5, 
    showcoastlines=True,
    coastlinecolor="black",
    showland=True,
    landcolor='lightgrey', 
    showocean=True,
    oceancolor='grey',
    showframe=True
)

for country in unique_countries:
    fig.add_trace(go.Choropleth(
        locations=[country],
        locationmode='country names',
        z=[1],
        colorscale=[[0, country_color_map[country]], [1, country_color_map[country]]],
        showscale=False
    ))

    fig.update_layout(
    legend=dict(
        font=dict(size=10),
        itemsizing='constant',
        title=dict(text="Страны (по убыванию города с max населением)")
    ),
    width=1200,
    height=700
)

In [34]:
fig


*Для красивого отображения и загрузки в гитверс:*

In [35]:
# import webbrowser
# fig.write_html('plotly.html')
# webbrowser.open('plotly.html')

#### Задание 2

**2.1** Топ-10 городов по населению

In [36]:
top_cities = df.nlargest(10, 'population')

fig1 = px.bar(
    top_cities,
    y='city',
    x='population',
    color='country',
    title='Топ-10 городов по населению',
    labels={'population': 'Население', 'city': 'Город'},
    text='population'
)

fig1.update_traces(texttemplate='%{text:.2s}', textposition='outside')
fig1.update_layout(yaxis={'categoryorder': 'total ascending'})
fig1.show()

**2.2** Количество городов в каждой стране (топ-10 стран по количеству крупных городов)

In [37]:
city_counts = df['country'].value_counts().reset_index()
city_counts.columns = ['country', 'count']

top_countries = city_counts.head(10)

fig2 = px.pie(
    top_countries,
    values='count',
    names='country',
    title='Количество городов в каждой стране (топ-10 стран по количеству крупных городов)',
    hole=0.3
)

fig2.update_traces(textposition='inside', textinfo='percent+label')
fig2.show()

**2.3** Распределение стран и городов

In [38]:
df_sun = df.copy()
df_sun['city_with_pop'] = df_sun['city'] + ' (' + (df_sun['population']/1000000).round(1).astype(str) + ' млн)'

fig3 = px.sunburst(
    df_sun,
    path=['country', 'city_with_pop'],
    values='population',
    title='Распределение стран и городов',
    color='country',
    color_discrete_map=country_color_map
)

fig3.update_layout(
    width=800,
    height=800
)

fig3.show()

### Вывод
Построены все требуемые графики.